In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset
import pandas as pd
import numpy as np
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer

# Cấu hình
model_name = "vinai/phobert-base"
train_path =  "D:\\N3\\HK2\\HS\\CK\\ABSA_VietNamese\\data\\processed\\sentiment_classification\\train.csv"
valid_path = "D:\\N3\\HK2\\HS\\CK\\ABSA_VietNamese\\data\\processed\\sentiment_classification\\dev.csv"
output_dir = "D:\\N3\\HK2\\HS\\CK\\ABSA_VietNamese\\checkpoints\\aspect_model"
max_length = 128

# Hàm tải dữ liệu
def load_dataframe(path):
    return pd.read_csv(path)

def build_dataset(df, tokenizer):
    texts = df["sentence"].astype(str).tolist()
    labels = [label_map[s] for s in df["sentiment"].tolist()]
    enc = tokenizer(texts, padding="max_length", truncation=True, max_length=max_length)
    enc["labels"] = labels
    return Dataset.from_dict(enc)

# Tải tokenizer và dữ liệu
tokenizer = AutoTokenizer.from_pretrained(model_name)
train_df = load_dataframe(train_path)
valid_df = load_dataframe(valid_path)

label_map = {'negative': 0, 'neutral': 1, 'positive': 2}
train_dataset = build_dataset(train_df, tokenizer)
valid_dataset = build_dataset(valid_df, tokenizer)
                        
# Tạo model
num_labels = len(label_map)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

# Thiết lập huấn luyện
training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    do_eval=True,
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    save_total_limit=2,
    seed=42,
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=preds, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorForTokenClassification(tokenizer)
)

...
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer)
)

trainer.train()
trainer.save_model(output_dir)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.bias                | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
C:\User

Step,Training Loss


KeyboardInterrupt: 